# S2: Station × Year Context Merge

Secondary merge per `MERGE.md` (§3, **S2**). Broadcasts every slow-moving /
annual covariate — land use, BMP adoption, NPDES point-source pressure, county
agriculture, and state chemical spending — onto each station via its watershed,
county, and state membership. Grain is **one row per station + year**.

**Year window:** 2015–2025 — the full WQ measurement window (all of S1 falls in
it) and the native span of P5/P6/P3b. P4 stops at 2024 and P3's crop/livestock
at 2025, so their trailing years land null; that's expected.

**Inputs** (all `data/03a_merge_primary/...`):
- `station-geo-soil.csv` (**P2**) — base station list + membership keys
  (`county_fips`, `huc12_code`, `huc8_code`). Crossed with the year range to
  form the grid.
- `huc12-landuse-bmp.csv` (**P5**) — CDL land use + HUC-8 BMP, joined on
  `huc12_code` + `year`.
- `npdes-facility.csv` (**P6**) — permit × fiscal-year facility context,
  **aggregated to `huc8` + year** here (sum/count of facilities, exceedances,
  violations, impairments), then joined on the station's `huc8_code` + `year`.
  HUC-8 is used rather than HUC-12 because every station's HUC-8 contains NPDES
  facilities (100% coverage vs. ~62% at HUC-12), giving a meaningful
  watershed-level point-source signal for every station.
- `county-agriculture.csv` (**P3**) — **only** the `cropyield__*` / `livestock__*`
  blocks (natively annual, dense 2015–2025), joined on `county_fips` + `year`.
- `county-agriculture-asof.csv` (**P3b**) — the `npfert__*` / `npmanure__*` blocks
  plus as-of provenance columns, joined on `county_fips` + `year`. P3b is the
  2015–2025-dense nutrient table; P3's raw quinquennial N&P is deliberately
  **not** used (only 2017 has a native value inside the window — see MERGE.md §3).
- `state-chemical-spending.csv` (**P4**) — statewide chemical application + farm
  spending, joined on `year` alone (IA-only, no spatial variation).

**Output:** `data/03b_merge_secondary/station-year-context.csv`, one row per
station + year.

In [1]:
import os

import pandas as pd

PRIMARY = "../../data/03a_merge_primary"
OUT_DIR = "../../data/03b_merge_secondary"
OUT_FILE = f"{OUT_DIR}/station-year-context.csv"

KEY = "MonitoringLocationIdentifier"
GRAIN = [KEY, "year"]
YEARS = list(range(2015, 2026))  # 2015-2025 inclusive, the WQ measurement window

# Codes stay strings so leading zeros survive.
STR_CODES = {"county_fips": str, "huc12_code": str, "huc10_code": str, "huc8_code": str}

## Step 1: Build the station × year grid from P2

P2 is one row per station. Keep only the identity + membership keys (the join
keys for the watershed/county layers below); the soil/geography detail already
lives in S1. Cross the stations with 2015–2025 to get the station-year base
grid.

In [2]:
p2 = pd.read_csv(f"{PRIMARY}/station-geo-soil.csv", dtype=STR_CODES)
assert not p2.duplicated(subset=[KEY]).any(), "P2 is not 1 row per station"

stations = p2[[KEY, "county_fips", "huc12_code", "huc8_code"]].copy()
grid = stations.merge(pd.DataFrame({"year": YEARS}), how="cross")

print(f"Stations: {len(stations):,} | years: {len(YEARS)} | grid: {grid.shape}")
assert not grid.duplicated(subset=GRAIN).any(), "Grid grain violated"
n_rows = len(grid)
df = grid

Stations: 1,666 | years: 11 | grid: (18326, 5)


## Step 2: P5 land use + BMP — join on `huc12_code` + `year`

P5 is one row per HUC-12 + year. Its `huc8_code` is dropped (the grid already
carries the station's `huc8_code`) so only the CDL `pct_*` and `bmp__*` columns
are added.

In [3]:
p5 = pd.read_csv(
    f"{PRIMARY}/huc12-landuse-bmp.csv",
    dtype={"huc12_code": str, "huc8_code": str},
).drop(columns=["huc8_code"])
assert not p5.duplicated(subset=["huc12_code", "year"]).any(), "P5 is not 1 row per huc12+year"

df = df.merge(p5, on=["huc12_code", "year"], how="left")
print(f"After P5: {df.shape} | land-use match: {df['pct_corn'].notna().mean():.1%}")
assert len(df) == n_rows, "P5 join fanned out station-year rows"

After P5: (18326, 20) | land-use match: 100.0%


## Step 3: P6 NPDES — aggregate to `huc8` + year, then join

P6 is one row per permit + fiscal year. Aggregate all permits in a HUC-8 for a
given year into watershed-level point-source pressure (facility count, summed
exceedances / violations / late reports, mean exceedance %, any 303(d)
impairment), then join on the station's `huc8_code` + `year`. The two
boolean-ish impairment flags are coerced to 0/1 first so they aggregate.

In [4]:
p6 = pd.read_csv(f"{PRIMARY}/npdes-facility.csv", dtype={"wbd_huc8": str})
p6["impaired_303d"] = p6["impaired_303d"].astype(float)
p6["attains_any_impaired"] = p6["attains_any_impaired"].map({True: 1.0, False: 0.0})

p6_huc8 = (
    p6.dropna(subset=["wbd_huc8"])
    .groupby(["wbd_huc8", "fiscal_year"])
    .agg(
        npdes__n_facilities=("npdes_id", "nunique"),
        npdes__n_impaired_303d=("impaired_303d", "sum"),
        npdes__any_impaired=("attains_any_impaired", "max"),
        npdes__dmr_n_records=("dmr_n_records", "sum"),
        npdes__dmr_n_exceedances=("dmr_n_exceedances", "sum"),
        npdes__dmr_n_violations=("dmr_n_violations", "sum"),
        npdes__dmr_n_late_reports=("dmr_n_late_reports", "sum"),
        npdes__dmr_mean_exceedence_pct=("dmr_mean_exceedence_pct", "mean"),
    )
    .reset_index()
    .rename(columns={"wbd_huc8": "huc8_code", "fiscal_year": "year"})
)
assert not p6_huc8.duplicated(subset=["huc8_code", "year"]).any(), "P6 aggregation not unique per huc8+year"

df = df.merge(p6_huc8, on=["huc8_code", "year"], how="left")
print(f"After P6: {df.shape} | NPDES huc8 match: {df['npdes__n_facilities'].notna().mean():.1%}")
assert len(df) == n_rows, "P6 join fanned out station-year rows"

After P6: (18326, 28) | NPDES huc8 match: 100.0%


## Step 4: P3 crop yields + livestock — join on `county_fips` + `year`

Pull **only** the `cropyield__*` / `livestock__*` blocks from P3 (the natively
annual, 2015–2025-dense columns). P3's raw quinquennial `npfert__*` /
`npmanure__*` / `manureinv__*` columns are left behind — the nutrient columns
come from P3b in the next step instead.

In [5]:
p3 = pd.read_csv(f"{PRIMARY}/county-agriculture.csv", dtype={"county_fips": str})
p3 = p3[p3["year"].isin(YEARS)]
ag_cols = [c for c in p3.columns if c.startswith(("cropyield__", "livestock__"))]
p3 = p3[["county_fips", "year"] + ag_cols]
assert not p3.duplicated(subset=["county_fips", "year"]).any(), "P3 is not 1 row per county+year"
print(f"P3 crop/livestock columns pulled: {len(ag_cols)}")

df = df.merge(p3, on=["county_fips", "year"], how="left")
print(f"After P3: {df.shape} | crop/livestock match: {df['cropyield__corn_grain__yield__survey'].notna().mean():.1%}")
assert len(df) == n_rows, "P3 join fanned out station-year rows"

P3 crop/livestock columns pulled: 57
After P3: (18326, 85) | crop/livestock match: 93.6%


## Step 5: P3b N&P fertilizer/manure (as-of) — join on `county_fips` + `year`

Pull the `npfert__*` / `npmanure__*` nutrient blocks plus the as-of provenance
columns (`np_base_year`, `np_years_stale`, refresh ratios/flags). P3b is dense
for 2015–2025 by construction, so this fills the nutrient columns for the whole
window.

In [6]:
p3b = pd.read_csv(f"{PRIMARY}/county-agriculture-asof.csv", dtype={"county_fips": str})
p3b = p3b[p3b["year"].isin(YEARS)]
PROVENANCE = [
    "np_base_year", "np_years_stale", "cattle_head_ratio", "hog_head_ratio",
    "manure_cattle_refreshed", "manure_hogs_refreshed",
]
nutrient_cols = [c for c in p3b.columns if c.startswith(("npfert__", "npmanure__"))]
p3b = p3b[["county_fips", "year"] + PROVENANCE + nutrient_cols]
assert not p3b.duplicated(subset=["county_fips", "year"]).any(), "P3b is not 1 row per county+year"
print(f"P3b nutrient columns pulled: {len(nutrient_cols)} (+{len(PROVENANCE)} provenance)")

df = df.merge(p3b, on=["county_fips", "year"], how="left")
print(f"After P3b: {df.shape} | N&P match: {df['npfert__n__total_kg'].notna().mean():.1%}")
assert len(df) == n_rows, "P3b join fanned out station-year rows"

P3b nutrient columns pulled: 16 (+6 provenance)
After P3b: (18326, 107) | N&P match: 100.0%


## Step 6: P4 state chemical spending — join on `year`

IA-only, so there's no spatial dimension — every station in a given year gets the
same statewide chemical-application and farm-spending values. `state_fips` is
dropped (constant `19`); the join is on `year` alone.

In [7]:
p4 = pd.read_csv(f"{PRIMARY}/state-chemical-spending.csv").drop(columns=["state_fips"])
assert not p4.duplicated(subset=["year"]).any(), "P4 is not 1 row per year"

df = df.merge(p4, on="year", how="left")
print(f"After P4: {df.shape} | state-spending match: {df['spend__feed__usd'].notna().mean():.1%}")
assert len(df) == n_rows, "P4 join fanned out station-year rows"

After P4: (18326, 217) | state-spending match: 90.9%


## Step 7: Final checks and save

In [8]:
print(f"Final shape: {df.shape}")
assert not df.duplicated(subset=GRAIN).any(), "Output grain violated: duplicate (station, year) rows"
assert len(df) == n_rows, "Row count changed vs. station-year grid"

print("\nCoverage (share of station-year rows):")
print(f"  P5 land use (huc12):    {df['pct_corn'].notna().mean():.1%}")
print(f"  P6 NPDES (huc8):        {df['npdes__n_facilities'].notna().mean():.1%}")
print(f"  P3 crop/livestock:      {df['cropyield__corn_grain__yield__survey'].notna().mean():.1%}")
print(f"  P3b N&P nutrients:      {df['npfert__n__total_kg'].notna().mean():.1%}")
print(f"  P4 state spending:      {df['spend__feed__usd'].notna().mean():.1%}")
df.head(3)

Final shape: (18326, 217)

Coverage (share of station-year rows):
  P5 land use (huc12):    100.0%
  P6 NPDES (huc8):        100.0%
  P3 crop/livestock:      93.6%
  P3b N&P nutrients:      100.0%
  P4 state spending:      90.9%


,MonitoringLocationIdentifier,county_fips,huc12_code,huc8_code,year,pct_corn,pct_soybean,pct_other_crops,pct_developed,pct_forest,...,spend__chemical_totals__usd,spend__chemical_totals__usd_per_operation,spend__feed__pct_of_operations,spend__feed__pct_of_prod_expenses,spend__feed__usd,spend__feed__usd_per_operation,spend__fertilizer_totals_incl_lime_and_soil_conditioners__pct_of_operations,spend__fertilizer_totals_incl_lime_and_soil_conditioners__pct_of_prod_expenses,spend__fertilizer_totals_incl_lime_and_soil_conditioners__usd,spend__fertilizer_totals_incl_lime_and_soil_conditioners__usd_per_operation
0,USGS-05387490,19191,070600020401,07060002,2015,0.3037,0.1255,0.0783,0.0716,0.0815,...,9.900000e+08,11314.0,39.2,18.7,5.190000e+09,59314.0,63.4,7.4,2.040000e+09,23314.0
1,USGS-05387490,19191,070600020401,07060002,2016,0.2783,0.1460,0.0903,0.0712,0.0796,...,1.130000e+09,12989.0,38.0,19.8,5.210000e+09,59885.0,60.4,7.1,1.880000e+09,21609.0
2,USGS-05387490,19191,070600020401,07060002,2017,0.3132,0.1419,0.0662,0.0713,0.0813,...,1.140000e+09,13240.0,40.9,16.7,4.400000e+09,51103.0,60.2,6.9,1.810000e+09,21022.0


In [9]:
os.makedirs(OUT_DIR, exist_ok=True)
df.to_csv(OUT_FILE, index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} cols -> {OUT_FILE}")

Saved 18,326 rows x 217 cols -> ../../data/03b_merge_secondary/station-year-context.csv
